In [2]:
#当前模型采用1层单向GRU结构，嵌入维度和隐藏层维度均为128，使用学习率0.001的Adam优化器训练20轮，配合0.3的dropout正则化和200的最大序列长度，在测试集上仅获得0.5858的准确率，表现接近随机猜测水平，表明模型容量严重不足且训练策略欠佳。
#最优模型应升级为1层双向GRU结合注意力机制的结构，将嵌入维度和隐藏层维度提升至256，并增加一个全连接层以增强特征提取能力。超参数方面需将学习率降至0.0005，dropout调整为0.2，训练轮数延长至30轮，最大序列长度扩展至300。优化方法应采用AdamW优化器配合权重衰减0.01，结合余弦退火重启的学习率调度策略，并加入梯度裁剪和标签平滑技术。经此系统性优化，模型准确率预计可从0.5858显著提升至0.75以上，完全满足0.7的性能要求。

In [15]:
from news_data import load_saved_data
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score
import re
import os
import sys

# ====================== 1. 加载并预处理数据 ======================
print("正在加载数据...")
try:
    data = load_saved_data("news_data.pkl")
    X_train = data["X_train"]
    X_test = data["X_test"]
    y_train = data["y_train"]
    y_test = data["y_test"]
    word_to_idx = data["word_to_idx"]
    vocab_size = data["vocab_size"]
    
    print(f"数据加载成功！")
    print(f"训练集大小: {len(X_train)}")
    print(f"测试集大小: {len(X_test)}")
    print(f"词汇表大小: {vocab_size}")
except Exception as e:
    print(f"加载数据失败: {e}")
    print("请确保 news_data.pkl 文件存在")
    sys.exit(1)

# 检查并清洗数据
def clean_sequence(seq):
    """清洗序列，确保所有元素都是整数"""
    cleaned = []
    for token in seq:
        if isinstance(token, (int, np.integer)):
            cleaned.append(int(token))
        elif isinstance(token, str):
            # 尝试转换字符串为整数
            try:
                cleaned.append(int(token))
            except:
                # 转换失败，使用<UNK>标记
                cleaned.append(1)  # 假设1是<UNK>的索引
        else:
            # 其他类型，使用<UNK>
            cleaned.append(1)
    return cleaned

# 处理训练集和测试集
print("正在清洗数据...")
X_train_clean = [clean_sequence(seq) for seq in X_train]
X_test_clean = [clean_sequence(seq) for seq in X_test]

# 分析序列长度
train_lengths = [len(seq) for seq in X_train_clean]
test_lengths = [len(seq) for seq in X_test_clean]

# 选择最大长度（覆盖90%的数据）
sorted_lengths = sorted(train_lengths)
percentile_90 = sorted_lengths[int(0.9 * len(sorted_lengths))]
max_len = min(200, percentile_90)  # 不超过200

print(f"序列长度统计:")
print(f"  训练集平均长度: {np.mean(train_lengths):.1f}")
print(f"  训练集最大长度: {max(train_lengths)}")
print(f"  选择最大长度: {max_len} (覆盖90%数据)")

# 序列填充函数
def pad_sequence(seq, max_len, padding_value=0):
    if len(seq) >= max_len:
        return seq[:max_len]
    else:
        return seq + [padding_value] * (max_len - len(seq))

# 填充序列
X_train_pad = [pad_sequence(seq, max_len) for seq in X_train_clean]
X_test_pad = [pad_sequence(seq, max_len) for seq in X_test_clean]

# 转换为PyTorch张量
X_train_tensor = torch.tensor(X_train_pad, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_pad, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# 划分验证集（从训练集中拆分20%作为验证集）
train_size = int(0.8 * len(X_train_tensor))
val_size = len(X_train_tensor) - train_size
X_train_final, X_val_final = X_train_tensor.split([train_size, val_size])
y_train_final, y_val_final = y_train_tensor.split([train_size, val_size])

print(f"\n数据集划分:")
print(f"  训练集: {len(X_train_final)}")
print(f"  验证集: {len(X_val_final)}")
print(f"  测试集: {len(X_test_tensor)}")

# 构建DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train_final, y_train_final)
val_dataset = TensorDataset(X_val_final, y_val_final)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ====================== 2. 定义1层GRU模型 ======================
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers=1, dropout=0.5):
        super(GRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(
            embedding_dim, 
            hidden_dim, 
            num_layers=n_layers, 
            batch_first=True, 
            dropout=dropout if n_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        # text: [batch_size, seq_len]
        embedded = self.dropout(self.embedding(text))  # [batch_size, seq_len, embedding_dim]
        gru_out, hidden = self.gru(embedded)  # gru_out: [batch_size, seq_len, hidden_dim]
        # 取最后一个时间步
        last_hidden = gru_out[:, -1, :]  # [batch_size, hidden_dim]
        last_hidden = self.dropout(last_hidden)
        return self.fc(last_hidden)  # [batch_size, output_dim]

# ====================== 3. 超参数设置与模型初始化 ======================
print("\n" + "="*50)
print("模型配置")
print("="*50)

embedding_dim = 128    # 词向量维度
hidden_dim = 128       # GRU隐藏层维度
output_dim = 1         # 二分类输出
n_layers = 1           # 1层GRU
dropout = 0.3          # dropout率

model = GRUClassifier(vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout)

# 计算模型参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

# 优化器与损失函数
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()  # 二分类交叉熵（带sigmoid）

# ====================== 4. 训练函数与验证函数 ======================
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for batch in loader:
        text, labels = batch
        text, labels = text.to(device), labels.to(device)
        
        optimizer.zero_grad()
        predictions = model(text).squeeze(1)  # [batch_size]
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # 计算准确率
        with torch.no_grad():
            preds = (torch.sigmoid(predictions) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return epoch_loss / len(loader), correct / total

def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in loader:
            text, labels = batch
            text, labels = text.to(device), labels.to(device)
            
            predictions = model(text).squeeze(1)
            loss = criterion(predictions, labels)
            epoch_loss += loss.item()
            
            preds = (torch.sigmoid(predictions) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return epoch_loss / len(loader), correct / total

# ====================== 5. 训练过程 ======================
print("\n" + "="*50)
print("开始训练")
print("="*50)

# 设备设置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"使用设备: {device}")

epochs = 20
best_val_acc = 0
patience = 5
counter = 0

for epoch in range(epochs):
    # 训练
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    
    # 验证
    val_loss, val_acc = evaluate_epoch(model, val_loader, criterion, device)
    
    print(f"Epoch {epoch+1:2d}/{epochs} | "
          f"训练 Loss: {train_loss:.4f} | 训练 Acc: {train_acc:.4f} | "
          f"验证 Loss: {val_loss:.4f} | 验证 Acc: {val_acc:.4f}")
    
    # 早停与保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'train_loss': train_loss,
        }, "best_gru_model.pth")
        print(f"  ✓ 保存最佳模型 (验证准确率: {val_acc:.4f})")
    else:
        counter += 1
        if counter >= patience:
            print(f"  ⚠ 验证准确率连续{patience}个epoch未提升，提前停止")
            break

# ====================== 6. 测试模型 ======================
print("\n" + "="*50)
print("测试模型")
print("="*50)

# 加载最佳模型
checkpoint = torch.load("best_gru_model.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

# 测试
model.eval()
test_loss, test_acc = evaluate_epoch(model, test_loader, criterion, device)

print(f"测试集结果:")
print(f"  测试损失: {test_loss:.4f}")
print(f"  测试准确率: {test_acc:.4f}")

# 检查是否达到目标
target_acc = 0.7
if test_acc > target_acc:
    print(f"\n✅ 成功！测试准确率 ({test_acc:.4f}) 高于目标值 {target_acc}")
else:
    print(f"\n⚠ 未达目标，测试准确率 ({test_acc:.4f}) 低于目标值 {target_acc}")
    print("建议调整以下超参数:")
    print("  1. 增加 embedding_dim 到 256")
    print("  2. 增加 hidden_dim 到 256")
    print("  3. 增加训练轮数到 30")
    print("  4. 调整 dropout 为 0.2 或 0.4")
    print("  5. 降低学习率为 0.0005")

# ====================== 7. 预测示例 ======================
print("\n" + "="*50)
print("预测示例")
print("="*50)

# 随机选择5个测试样本
sample_indices = np.random.choice(len(X_test_tensor), min(5, len(X_test_tensor)), replace=False)

model.eval()
with torch.no_grad():
    for i, idx in enumerate(sample_indices):
        sample = X_test_tensor[idx:idx+1].to(device)
        true_label = y_test_tensor[idx].item()
        
        prediction = model(sample).squeeze(1)
        pred_prob = torch.sigmoid(prediction).item()
        pred_label = 1 if pred_prob > 0.5 else 0
        
        correct = "✓" if pred_label == true_label else "✗"
        print(f"样本 {i+1}:")
        print(f"  真实标签: {true_label} | 预测标签: {pred_label}")
        print(f"  预测概率: {pred_prob:.4f} | 结果: {correct}")
        print()

# ====================== 8. 保存完整模型 ======================
print("\n保存完整模型信息...")
torch.save({
    'model_state_dict': model.state_dict(),
    'word_to_idx': word_to_idx,
    'vocab_size': vocab_size,
    'max_len': max_len,
    'embedding_dim': embedding_dim,
    'hidden_dim': hidden_dim,
    'test_accuracy': test_acc
}, "gru_classifier_complete.pth")

print("模型已保存:")
print("  - best_gru_model.pth (训练检查点)")
print("  - gru_classifier_complete.pth (完整模型)")
print("\n训练完成！")

正在加载数据...
数据加载成功！
训练集大小: 1079
测试集大小: 717
词汇表大小: 11753
正在清洗数据...
序列长度统计:
  训练集平均长度: 1258.7
  训练集最大长度: 45731
  选择最大长度: 200 (覆盖90%数据)

数据集划分:
  训练集: 863
  验证集: 216
  测试集: 717

模型配置
模型总参数量: 1,603,585
可训练参数量: 1,603,585

开始训练
使用设备: cpu
Epoch  1/20 | 训练 Loss: 0.7100 | 训练 Acc: 0.5446 | 验证 Loss: 0.6864 | 验证 Acc: 0.5972
  ✓ 保存最佳模型 (验证准确率: 0.5972)
Epoch  2/20 | 训练 Loss: 0.6838 | 训练 Acc: 0.5944 | 验证 Loss: 0.6824 | 验证 Acc: 0.5972
Epoch  3/20 | 训练 Loss: 0.6818 | 训练 Acc: 0.5991 | 验证 Loss: 0.6775 | 验证 Acc: 0.6019
  ✓ 保存最佳模型 (验证准确率: 0.6019)
Epoch  4/20 | 训练 Loss: 0.6899 | 训练 Acc: 0.5620 | 验证 Loss: 0.6811 | 验证 Acc: 0.5972
Epoch  5/20 | 训练 Loss: 0.6806 | 训练 Acc: 0.5968 | 验证 Loss: 0.6844 | 验证 Acc: 0.5972
Epoch  6/20 | 训练 Loss: 0.6850 | 训练 Acc: 0.5794 | 验证 Loss: 0.6845 | 验证 Acc: 0.5972
Epoch  7/20 | 训练 Loss: 0.6819 | 训练 Acc: 0.5886 | 验证 Loss: 0.6814 | 验证 Acc: 0.5972
Epoch  8/20 | 训练 Loss: 0.6757 | 训练 Acc: 0.5968 | 验证 Loss: 0.6902 | 验证 Acc: 0.5972
  ⚠ 验证准确率连续5个epoch未提升，提前停止

测试模型
测试集结果:
  测试损失: 0.6802
  测试准